# Video and voiceover to a lip-synced clip

Run the cells from top to bottom. This notebook creates **one five-second video** using your Magic Hour account and existing credits. Each rerun of the generation cell creates another project.

[Create an API key](https://magichour.ai/developer) · [Full recipe and recovery instructions](https://docs.magichour.ai/get-started/starter-recipes)

Use media you own or have permission to use. Files selected below are uploaded to this Google Colab runtime and then to Magic Hour. Review generated output before publishing. This notebook uses your supplied voiceover; it does not translate or generate speech.


In [ ]:
%pip install -q "magic-hour==0.78.1"


In [ ]:
import os
from getpass import getpass

os.environ["MAGIC_HOUR_API_KEY"] = getpass("Magic Hour API key (hidden): ").strip()
if not os.environ["MAGIC_HOUR_API_KEY"]:
    raise ValueError("Enter an API key before continuing.")


## Upload your input files


In [ ]:
from google.colab import files
from pathlib import Path

print("Upload one MP4 video with a clearly visible face, at least five seconds long.")
uploaded_video = files.upload()
if len(uploaded_video) != 1:
    raise ValueError("Upload exactly one video before continuing.")
video_filename = next(iter(uploaded_video))
if Path(video_filename).suffix.lower() != ".mp4":
    raise ValueError("Use an MP4 video.")

print("Upload one MP3 voiceover, at least five seconds long.")
uploaded_audio = files.upload()
if len(uploaded_audio) != 1:
    raise ValueError("Upload exactly one audio file before continuing.")
audio_filename = next(iter(uploaded_audio))
if Path(audio_filename).suffix.lower() != ".mp3":
    raise ValueError("Use an MP3 audio file.")


## Create one video

This cell uses credits. Save the printed project ID. If polling or downloading fails, use the recovery instructions in the linked guide instead of rerunning this cell. Stopping Colab does not cancel a submitted project.


In [ ]:
import os
from pathlib import Path

from magic_hour import Client

video = Path(video_filename)
audio = Path(audio_filename)
for source in (video, audio):
    if not source.is_file():
        raise SystemExit(f"Add {source} beside this script before running.")

client = Client(token=os.environ["MAGIC_HOUR_API_KEY"])
os.environ.setdefault("MAGIC_HOUR_POLL_INTERVAL", "3")

uploaded_video = client.v1.files.upload_file(file=str(video))
uploaded_audio = client.v1.files.upload_file(file=str(audio))
job = client.v1.lip_sync.create(
    assets={
        "video_source": "file",
        "video_file_path": uploaded_video,
        "audio_file_path": uploaded_audio,
    },
    start_seconds=0,
    end_seconds=5,
    name="Lip sync starter recipe",
)
print(f"Project ID: {job.id}", flush=True)

Path("outputs/lip-sync").mkdir(parents=True, exist_ok=True)
result = client.v1.video_projects.check_result(
    id=job.id,
    wait_for_completion=True,
    download_outputs=True,
    download_directory="outputs/lip-sync",
)
if result.status != "complete" or not result.downloaded_paths:
    raise SystemExit(f"Project {result.id}: {result.status}; {result.error}")

for path in result.downloaded_paths:
    print(f"Downloaded: {path}")


## Preview and save

Colab storage is temporary. Download the completed video to keep it. Clear all cell outputs before sharing a copy of this notebook; outputs can contain your media and project ID. The API key is entered with a hidden prompt and is not stored in notebook source.


In [ ]:
from IPython.display import Video, display
from google.colab import files

for output_path in result.downloaded_paths:
    display(Video(output_path, embed=True))
    files.download(output_path)
